In [ ]:
#CÉLULA 1: IMPORTS
import numpy as np 
import pandas as pd
from src.mggp import MGGP
from sklearn.preprocessing import StandardScaler
import os
import re 
import gc
import shutil
import pickle
import dill 
from scipy import signal
import matplotlib.pyplot as plt
# from tensorflow.keras.models import load_model
print("importadas!")

In [ ]:
# CÉLULA 2: FUNÇÕES AUXILIARES, PREPARAÇÃO DE DADOS E CONFIGURAÇÃO
# =============================================================================

# --- INSIRA AQUI O NÚMERO DO TREINAMENTO ---
NUMERO_TREINAMENTO = 30 
# -------------------------------------------

# =============================================================================
# 1. CARREGAMENTO E DESLOCAMENTO DOS DADOS (OFFSET NO Y)
# =============================================================================
OFFSET_Y = 10.0 # Deslocamento vertical para afastar o y do zero e proteger o MAPE

def carregar_dados_bicicleta(caminho_arquivo):
    """
    Carrega o arquivo excel, separa as entradas (u) e saídas (y), e aplica
    um deslocamento vertical (offset) apenas na saída 'y' para evitar divisões
    por zero no cálculo do MAPE.
    """
    df = pd.read_excel(caminho_arquivo, header=None)
    total_original = len(df)
    
    # Extração bruta
    u = df.iloc[:, [14, 2, 22, 23, 11]].values
    y_raw = df.iloc[:, [18, 17]].values
    
    # Aplicação do Deslocamento em Y (y_novo = y_original + 10)
    y_deslocado = y_raw + OFFSET_Y
        
    return u, y_deslocado

def reverter_deslocamento(y_pred):
    """
    Remove o deslocamento vertical (offset) após a previsão do modelo,
    trazendo a curva de volta para o eixo zero original.
    """
    return y_pred - OFFSET_Y

# =============================================================================
# 2. FERRAMENTAS DE ANÁLISE DE INÉRCIA 
# =============================================================================
def analyze_cross_correlations(u_train, y_train, max_lag=50):    
    u_df = pd.DataFrame(u_train, columns=['ax', 'ay', 'w_front', 'w_rear', 'yaw_rate'])
    y_df = pd.DataFrame(y_train, columns=['vx', 'vy'])
    results = {}
    for output_idx, output_col in enumerate(y_df.columns):
        output_signal = y_df[output_col].values
        for input_idx, input_col in enumerate(u_df.columns):
            input_signal = u_df[input_col].values
            correlation = signal.correlate(output_signal, input_signal, mode='full', method='auto')
            norm_factor = np.sqrt(np.sum(input_signal**2) * np.sum(output_signal**2))
            correlation_normalized = correlation / norm_factor if norm_factor > 0 else correlation
            lags = signal.correlation_lags(len(output_signal), len(input_signal), mode='full')
            peak_idx = np.argmax(np.abs(correlation_normalized))
            key = f"Output_{output_col}_Input_{input_col}"
            results[key] = {'peak_lag': lags[peak_idx], 'peak_value': correlation_normalized[peak_idx], 'correlation': correlation_normalized, 'lags': lags}
    return results

def plot_top_correlations(results, u_train, y_train, top_n=10):    
    u_col_names = ['ax', 'ay', 'w_front', 'w_rear', 'yaw_rate']
    y_col_names = ['vx', 'vy']
    sorted_results = sorted(results.items(), key=lambda x: abs(x[1]['peak_value']), reverse=True)
    top_results = sorted_results[:top_n]
    
    fig, axes = plt.subplots(2, 1, figsize=(12, 10))
    names = [f"{key.split('_Input_')[1]}\n→{key.split('Output_')[1].split('_Input_')[0]}" for key, _ in top_results]
    values = [result['peak_value'] for _, result in top_results]
    lags = [result['peak_lag'] for _, result in top_results]
    
    bars = axes[0].bar(range(len(names)), values, color='skyblue', alpha=0.7)
    axes[0].set_title(f'Top {top_n} Correlações Cruzadas (Inércia da Bicicleta)')
    axes[0].set_xticks(range(len(names)))
    axes[0].set_xticklabels(names, rotation=45, ha='right')
    axes[0].grid(True, alpha=0.3)
    
    for i, (bar, lag) in enumerate(zip(bars, lags)):
        axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height(), f'{bar.get_height():.3f}\n(lag:{lag})', ha='center', va='bottom', fontsize=8)
    
    correlation_matrix = np.zeros((len(y_col_names), len(u_col_names)))
    for output_idx, output_col in enumerate(y_col_names):
        for input_idx, input_col in enumerate(u_col_names):
            key = f"Output_{output_col}_Input_{input_col}"
            if key in results:
                correlation_matrix[output_idx, input_idx] = results[key]['peak_value']
    
    im = axes[1].imshow(correlation_matrix, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
    axes[1].set_title('Mapa de Calor - Entradas vs Saídas')
    axes[1].set_xticks(range(len(u_col_names)))
    axes[1].set_yticks(range(len(y_col_names)))
    axes[1].set_xticklabels(u_col_names, fontsize=10)
    axes[1].set_yticklabels(y_col_names, fontsize=10)
    plt.colorbar(im, ax=axes[1])
    plt.tight_layout()
    plt.show()

# =============================================================================
# 3. CONFIGURAÇÃO DO EXPERIMENTO E CRIAÇÃO DAS PASTAS
# =============================================================================
DIRETORIO_BASE = r"C:\IC_alex\TreinamentoMGGP_2_0\RESULTADOS"
pasta_salvamento = os.path.join(DIRETORIO_BASE, f"Treinamento_{NUMERO_TREINAMENTO}")
os.makedirs(pasta_salvamento, exist_ok=True)

parametros_mggp = {
    "nDelays": [1,2,5,10],
    "generations": 300,            
    "populationSize": 300,
    "evaluationMode": "MAPE",
    "k": 300, #testar com 500
    "evaluationType": "MShooting",
    "evaluationTypeTest": "FreeRun",
    "nTerms": 1,
    "maxHeight": 7,                        
    "mutationRate": 0.3,
    "crossoverRate": 0.8,
    "elitePercentage": 10,
    "mode": "MIMO",
    "froe_mode": False,                   
    "operators": ['add', 'subtraction', 'mul']    
}

df_params = pd.DataFrame(list(parametros_mggp.items()), columns=['Parametro', 'Valor'])
caminho_params = os.path.join(pasta_salvamento, "parametros_utilizados.csv")
df_params.to_csv(caminho_params, index=False)

print("\n✅ Célula 2 carregada! Offset de Y ativado (Y_novo = Y_original + 10).")

In [ ]:
# CÉLULA 3: FÁBRICA DE IAs (GRÁFICOS FORÇADOS E SALVAMENTO COM DILL)
# =============================================================================

print("--- 1. Lendo os arquivos Excel ---")
# Aqui você deve colocar o caminho COMPLETO do arquivo .xlsx
caminho_treino = r'C:\IC_alex\Database\wang21dv_bic_MGGP.xlsx'
caminho_validacao = r'C:\IC_alex\Database\wang22dv_bic_MGGP.xlsx'

# A função agora só recebe o caminho! O offset de +10 já vem embutido da Célula 2
u_train, y_train = carregar_dados_bicicleta(caminho_treino)
u_val, y_val = carregar_dados_bicicleta(caminho_validacao)

print("\n✅ Dados carregados com Offset de +10 no Y para proteção do MAPE.")

# --- 2. Configuração da Fábrica ---
pasta_training = os.path.join(pasta_salvamento, "TRAINING")
os.makedirs(pasta_training, exist_ok=True)

meta_modelos = 10        
modelos_aprovados = 0
tentativas = 0
max_tentativas = 50 # Trava de segurança

print(f"\n Buscando {meta_modelos} modelos com MAPE < 100%...")

while modelos_aprovados < meta_modelos and tentativas < max_tentativas:
    tentativas += 1
    print(f"\n" + "="*50)
    print(f"⚙️ Iniciando Tentativa {tentativas}/{max_tentativas} (Aprovados: {modelos_aprovados}/{meta_modelos})")
    print("="*50)

    caminho_temp = os.path.join(pasta_salvamento, "temp_mggp.pkl")
    
    mggp = MGGP(
        inputs=u_train, outputs=y_train,
        validation=(u_val, y_val),
        filename=caminho_temp,
        **parametros_mggp
    )
    
    mggp.run()
    
    yp, yd = None, None
    erro_v = 999.0
    
    best_model = mggp._hof[0]
    if parametros_mggp["evaluationTypeTest"] != "MShooting":
        args_val = (y_val, u_val)
    else:
        args_val = (parametros_mggp["k"], y_val, u_val)
        
    # =====================================================================
    # PASSO 1: GERAR PREVISÕES 
    # =====================================================================
    try:
        yp, yd = best_model.predict(parametros_mggp["evaluationTypeTest"], *args_val)
    except Exception as e:
        print(f" PREDICT FALHOU! O modelo não conseguiu nem calcular a linha. Erro: {e}")
        
    # =====================================================================
    # PASSO 2: FORÇAR O GRÁFICO NA TELA (MANTENDO O OFFSET)
    # =====================================================================
    if yp is not None and yd is not None:
        try:
            limite = len(yd) 
            
            fig, axs = plt.subplots(2, 1, figsize=(10, 6))
            fig.suptitle(f"Tentativa {tentativas} - Resultado da Equação (Offset +10)", fontsize=14)
            
            # Plotando diretamente o yd e yp (que já estão com o offset de +10)
            axs[0].plot(yd[:limite, 0], label="Real (+10)", color="navy")
            axs[0].plot(yp[:limite, 0], label="MGGP (+10)", color="orange", linestyle="--")
            axs[0].set_title(f"Saída 1 (vx)")
            axs[0].legend(loc="upper right")
            axs[0].grid(True, alpha=0.3)
            
            axs[1].plot(yd[:limite, 1], label="Real (+10)", color="darkgreen")
            axs[1].plot(yp[:limite, 1], label="MGGP (+10)", color="red", linestyle="--")
            axs[1].set_title(f"Saída 2 (vy)")
            axs[1].legend(loc="upper right")
            axs[1].grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show() 
        except Exception as e_grafico:
            print(f"⚠️ Erro ao desenhar o gráfico: {e_grafico}")

        # =====================================================================
        # PASSO 3: CALCULAR O MAPE
        # =====================================================================
        try:
            erro_v = best_model.score(yd, yp, parametros_mggp["evaluationMode"])
            print(f" MAPE Calculado (com offset): {erro_v:.2f}%")
        except Exception as e:
            print(f" SCORE FALHOU! Divisão por zero ou valor infinito no MAPE. Erro: {e}")

    # =====================================================================
    # AVALIAÇÃO FINAL DA TENTATIVA
    # =====================================================================
    if 0.0 < erro_v < 100.0: 
        nome_modelo = f"modelo_mape_{modelos_aprovados + 1}.pkl"
        caminho_final = os.path.join(pasta_training, nome_modelo)
        
        with open(caminho_final, 'wb') as f:
            dill.dump(best_model, f)
            
        modelos_aprovados += 1
        print(f"✅ APROVADO! Modelo salvo como {nome_modelo}.\n")
        
    else:
        print(f" REPROVADO (MAPE ruim ou inválido).\n")
        
    if os.path.exists(caminho_temp): 
        os.remove(caminho_temp)
    
    del mggp
    gc.collect()

if modelos_aprovados == meta_modelos:
    print(f"\n SUCESSO! {meta_modelos} modelos salvos na pasta TRAINING.")
else:
    print(f"\n PARADA: Limite de {max_tentativas} tentativas atingido. Modelos salvos: {modelos_aprovados}/{meta_modelos}.")

In [ ]:
# CÉLULA 4: VALIDAÇÃO EM MASSA LIMPA (COM DILL) + CSV
# =============================================================================

def calcular_mape(y_true, y_pred):
    y_t = np.where(np.abs(y_true) < 1e-6, 1e-6, y_true)
    return np.mean(np.abs((y_t - y_pred) / y_t)) * 100

pistas = {
    "Wang 2.1 (Treino)": r'C:\IC_alex\Database\wang21dv_bic_MGGP.xlsx',
    "Wang 2.2 (Validação)": r'C:\IC_alex\Database\wang22dv_bic_MGGP.xlsx',
    "Wang 3.1 (Teste A)": r'C:\IC_alex\Database\wang31dv_bic_MGGP.xlsx',
    "Wang 8.1 (Teste B)": r'C:\IC_alex\Database\wang81dv_bic_MGGP.xlsx'
}

pasta_training = os.path.join(pasta_salvamento, "TRAINING")
arquivos_modelos = [f for f in os.listdir(pasta_training) if f.endswith('.pkl')]

print(f"🔎 Encontrados {len(arquivos_modelos)} modelos para validação.")
resultados_csv = []

if len(arquivos_modelos) > 0:
    for idx_mod, nome_arquivo in enumerate(arquivos_modelos):
        caminho_modelo = os.path.join(pasta_training, nome_arquivo)
        print(f"\n--- Validando {nome_arquivo} ({idx_mod+1}/{len(arquivos_modelos)}) ---")
        
        try:
            # 1. CARREGAMENTO DIRETO E LIMPO COM DILL
            with open(caminho_modelo, 'rb') as f:
                modelo_real = dill.load(f)
        except Exception as e:
            print(f"   🚨 Erro ao carregar modelo: {e}")
            continue 
            
        fig, axes = plt.subplots(4, 2, figsize=(18, 20))
        fig.suptitle(f"Validação Completa (Offset +10): {nome_arquivo}", fontsize=20, y=1.02)
        
        for idx_pista, (nome_pista, caminho_pista) in enumerate(pistas.items()):
            # Removido o aplicar_filtro. A função agora já traz o dado com o +10 embutido!
            u_val_raw, y_val_raw = carregar_dados_bicicleta(caminho_pista)
            
            try:
                # 2. PREVISÃO DIRETA
                if parametros_mggp["evaluationTypeTest"] != "MShooting":
                    args_val = (y_val_raw, u_val_raw)
                else:
                    args_val = (parametros_mggp["k"], y_val_raw, u_val_raw)
                    
                yp_raw, yd_raw = modelo_real.predict(parametros_mggp["evaluationTypeTest"], *args_val)
                
                min_len = min(len(yd_raw), len(yp_raw))
                yd_raw, yp_raw = yd_raw[:min_len], yp_raw[:min_len]
                
                # MAPE é calculado com os dados deslocados para não explodir em infinito
                mape_vx = calcular_mape(yd_raw[:, 0], yp_raw[:, 0])
                mape_vy = calcular_mape(yd_raw[:, 1], yp_raw[:, 1])
                mape_medio = (mape_vx + mape_vy) / 2
                
            except Exception as e:
                print(f"   🚨 Erro na pista {nome_pista}: {e}")
                yp_raw, yd_raw = np.zeros_like(y_val_raw), y_val_raw
                mape_vx, mape_vy, mape_medio = 999.9, 999.9, 999.9
            
            resultados_csv.append({
                "Modelo": nome_arquivo,
                "Pista": nome_pista,
                "MAPE_Vx (%)": round(mape_vx, 2), 
                "MAPE_Vy (%)": round(mape_vy, 2), 
                "MAPE_Médio (%)": round(mape_medio, 2)
            })

            # Gráficos (Avisando na legenda que é o dado deslocado)
            axes[idx_pista, 0].plot(yd_raw[:, 0], color='black', label='Real (+10)')
            axes[idx_pista, 0].plot(yp_raw[:, 0], color='blue', linestyle='--', label='IA (+10)')
            axes[idx_pista, 0].set_title(f"{nome_pista} - Vx | MAPE: {mape_vx:.2f}%")
            if idx_pista == 0: axes[idx_pista, 0].legend()

            axes[idx_pista, 1].plot(yd_raw[:, 1], color='black', label='Real (+10)')
            axes[idx_pista, 1].plot(yp_raw[:, 1], color='red', linestyle='--', label='IA (+10)')
            axes[idx_pista, 1].set_title(f"{nome_pista} - Vy | MAPE: {mape_vy:.2f}%")
            if idx_pista == 0: axes[idx_pista, 1].legend()

        plt.tight_layout()
        plt.savefig(os.path.join(pasta_salvamento, nome_arquivo.replace(".pkl", ".png")), bbox_inches='tight')
        plt.close(fig) 
        print(f"   ✅ Gráfico salvo!")

    # CSV
    if len(resultados_csv) > 0:
        pd.DataFrame(resultados_csv).to_csv(os.path.join(pasta_salvamento, "relatorio_validacao.csv"), index=False, sep=';', decimal=',')
        print("\n📊 Tabela CSV criada com sucesso!")

print("\n" + "🏁"*20 + "\nVALIDAÇÃO FINALIZADA!\n" + "🏁"*20)

In [ ]:
# CÉLULA: LIMPEZA PROFUNDA DE MEMÓRIA RAM
# =============================================================================
import gc
import matplotlib.pyplot as plt

print("🧹 Iniciando faxina na memória RAM...")

# 1. FECHAR GRÁFICOS RESIDUAIS
plt.close('all')
print("   ✅ Gráficos e figuras em segundo plano fechados.")

# 2. FORÇAR O COLETOR DE LIXO (GARBAGE COLLECTOR)
objetos_removidos = gc.collect()
print(f"   ✅ Coletor de lixo executado (aprox. {objetos_removidos} objetos órfãos removidos).")

# -----------------------------------------------------------------------------
try:
    del u_train, y_train, u_val, y_val
    gc.collect()
    print("   ✅ Matrizes de dados destruídas.")
except NameError:
     pass

print("\n✨ Memória RAM otimizada com sucesso!")